# ViFake Analytics — Exploratory Data Analysis

**Dataset:** Vietnamese child scam detection (synthetic, privacy-safe)  
**Model pipeline:** PhoBERT (NLP) + CLIP (Vision) + XGBoost Fusion  
**Compliance:** Nghị định 13/2023/NĐ-CP — 100% synthetic, no real user data

---

In [1]:
import json
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Style ────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans',
})
PALETTE = ['#534AB7', '#1D9E75', '#D85A30', '#D4537E', '#BA7517', '#3B8BD4']
sns.set_palette(PALETTE)

DATA_DIR = Path('../data/synthetic')

# ── Load data — merge JSON (rich metadata) + JSONL (large, balanced) ──────
def to_df(records, split):
    rows = []
    for r in records:
        # Normalise age_group: JSONL uses 'age_focus', JSON uses 'age_group'
        age = r.get('age_group') or r.get('age_focus', 'unknown')
        # Normalise scenario: some JSONL records may not have it
        scenario_raw = r.get('scenario', 'unknown')
        # Normalise label: 0=SAFE,1=FAKE_SCAM; or label_str
        label_str = r.get('label_str', '')
        if not label_str:
            label_str = 'SAFE' if r.get('label', 1) == 0 else 'FAKE_SCAM'
        meta = r.get('metadata', {})
        lang_variant = meta.get('language_variant', r.get('language_variant', 'unknown'))
        rows.append({
            'split': split,
            'text': r.get('text', ''),
            'label': r.get('label', 1),
            'label_str': label_str,
            'scenario': scenario_raw,
            'severity': r.get('severity', 'medium'),
            'age_group': age,
            'realism_score': float(r.get('realism_score', 0.85)),
            'asks_for_credentials': r.get('risk_indicators', {}).get('asks_for_credentials', False),
            'includes_fake_link': r.get('risk_indicators', {}).get('includes_fake_link', False),
            'creates_urgency': r.get('risk_indicators', {}).get('creates_urgency', False),
            'language_variant': lang_variant,
        })
    return pd.DataFrame(rows)

# Primary source: JSONL (2800 balanced samples — newest generator)
jsonl_records = []
jsonl_path = DATA_DIR / 'phobert_train.jsonl'
if jsonl_path.exists():
    with open(jsonl_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                # JSONL schema: text, label, label_str, scenario (may be absent)
                jsonl_records.append(rec)
    print(f'JSONL loaded: {len(jsonl_records)} records')

# Secondary: rich JSON (750 scam samples with full metadata fields)
train_raw = json.loads((DATA_DIR / 'phobert_train.json').read_text())
val_raw   = json.loads((DATA_DIR / 'phobert_val.json').read_text())
report    = json.loads((DATA_DIR / 'labeled_dataset_report.json').read_text())

# Prefer JSONL as primary frame (richer class balance)
df_jsonl = to_df(jsonl_records, 'train') if jsonl_records else pd.DataFrame()
df_json  = to_df(train_raw, 'train')
df_val   = to_df(val_raw, 'val')

# Merge, dropping duplicates on text
df = pd.concat([df_jsonl, df_json, df_val], ignore_index=True)
df = df.drop_duplicates(subset='text').reset_index(drop=True)

print(f'Total unique records: {len(df):,}')
print(f'  Label distribution: {df.label_str.value_counts().to_dict()}')
print(f'  Scenarios: {df.scenario.nunique()}')
print(f'  Age groups: {df.age_group.nunique()}')

JSONL loaded: 2800 records
Total unique records: 2,903
  Label distribution: {'FAKE_SCAM': 2319, 'SAFE': 584}
  Scenarios: 7
  Age groups: 4


## 1 — Dataset Overview

In [2]:
overview = pd.DataFrame([
    ['Total samples', f"{len(df):,}"],
    ['Train split', f"{(df.split=='train').sum():,}"],
    ['Val split', f"{(df.split=='val').sum():,}"],
    ['Scam scenarios', f"{df.scenario.nunique()}"],
    ['Age groups covered', f"{df.age_group.nunique()}"],
    ['Avg realism score', f"{df.realism_score.mean():.3f}"],
    ['Avg safety score', '1.000  (all synthetic)'],
    ['Generation date', report['dataset_info']['generation_date'][:10]],
    ['Language', 'Vietnamese (teencode + formal)'],
    ['Compliance', 'Nghị định 13/2023/NĐ-CP'],
], columns=['Metric', 'Value'])

overview.style.hide(axis='index').set_caption('Dataset Overview')

Metric,Value
Total samples,"2,903"
Train split,"2,851"
Val split,52
Scam scenarios,7
Age groups covered,4
Avg realism score,0.858
Avg safety score,1.000 (all synthetic)
Generation date,2026-05-04
Language,Vietnamese (teencode + formal)
Compliance,Nghị định 13/2023/NĐ-CP


## 2 — Scenario Distribution

In [3]:
scenario_counts = df.groupby('scenario').size().sort_values(ascending=True)
scenario_labels = {
    'account_theft': 'Account Theft\n(credential harvest)',
    'crypto_scam': 'Crypto Scam\n(USDT/ETH/MetaMask)',
    'robux_phishing': 'Robux Phishing\n(game reward fraud)',
    'gift_card_scam': 'Gift Card Scam\n(iTunes/Google Play)',
    'malicious_link': 'Malicious Link\n(drive-by phishing)',
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(
    [scenario_labels.get(s, s) for s in scenario_counts.index],
    scenario_counts.values,
    color=PALETTE[:len(scenario_counts)],
    edgecolor='white', linewidth=0.5,
)
for bar, val in zip(bars, scenario_counts.values):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2,
            f'{val} ({val/len(df)*100:.1f}%)', va='center', fontsize=9)
ax.set_xlabel('Number of samples')
ax.set_title('Scam Scenario Distribution (train + val)', fontsize=13, fontweight='bold')
ax.set_xlim(0, scenario_counts.max() * 1.25)
plt.tight_layout()
plt.show()
print(scenario_counts.to_string())

scenario
legitimate_conversation       8
account_theft                10
crypto_scam                  10
gift_card_scam               10
malicious_link              143
robux_phishing              152
unknown                    2570


## 3 — Age Group Distribution

In [4]:
age_counts = df.groupby('age_group').size().reindex(['8-10', '11-13', '14-17'])
age_labels = {'8-10': '8–10 tuổi\n(tiểu học)', '11-13': '11–13 tuổi\n(THCS sớm)', '14-17': '14–17 tuổi\n(THCS/THPT)'}

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(
    [age_labels[a] for a in age_counts.index],
    age_counts.values,
    color=['#D85A30', '#534AB7', '#1D9E75'],
    edgecolor='white', linewidth=0.5, width=0.5,
)
for bar, val in zip(bars, age_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 3,
            f'{val}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Samples')
ax.set_title('Distribution by Target Age Group', fontsize=13, fontweight='bold')
ax.set_ylim(0, age_counts.max() * 1.2)
plt.tight_layout()
plt.show()

## 4 — Severity × Scenario Stacked Bar

In [5]:
sev_x_scen = df.groupby(['scenario', 'severity']).size().unstack(fill_value=0)
sev_x_scen = sev_x_scen.reindex(['account_theft','crypto_scam','robux_phishing','gift_card_scam','malicious_link'])

fig, ax = plt.subplots(figsize=(9, 4))
bottom = np.zeros(len(sev_x_scen))
colors_sev = {'high': '#D85A30', 'medium': '#534AB7'}
for sev, color in colors_sev.items():
    if sev in sev_x_scen.columns:
        vals = sev_x_scen[sev].values
        bars = ax.bar(sev_x_scen.index, vals, bottom=bottom, label=sev.capitalize(), color=color, edgecolor='white')
        for bar, v, b in zip(bars, vals, bottom):
            if v > 10:
                ax.text(bar.get_x() + bar.get_width()/2, b + v/2, str(v),
                        ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        bottom += vals

ax.set_xticks(range(len(sev_x_scen)))
ax.set_xticklabels(sev_x_scen.index, rotation=20, ha='right')
ax.set_ylabel('Samples')
ax.set_title('Severity Distribution per Scenario', fontsize=13, fontweight='bold')
ax.legend(title='Severity')
plt.tight_layout()
plt.show()

## 5 — Top Scam Keywords (from real text)

In [6]:
STOPWORDS = {
    'và','là','của','có','trong','không','được','để','này','đó','ở','với','khi','từ',
    'cho','đến','ra','về','thì','mà','nên','hay','vì','nhưng','tại','rồi','ơi',
    'cần','một','các','còn','đã','đang','sẽ','bị','cũng','như','theo','lên',
    'xuống','nếu','thôi','nha','nè','vào','tại','link','này','đây','đó',
    'acc','bạn','mình','minh','khoa','linh','tuan','quynh','t','m','ae','bạn',
}

all_words = []
for text in df['text']:
    words = re.findall(r'[a-zA-ZÀ-ỹ]{2,}', text.lower())
    all_words.extend([w for w in words if w not in STOPWORDS and len(w) > 1])

word_freq = Counter(all_words)
top_words = word_freq.most_common(25)
words, counts = zip(*top_words)

# Color by risk category
HIGH_RISK = {'verify','hack','pass','credentials','malicious','phishing','scam','theft'}
MED_RISK  = {'robux','usdt','metamask','giveaway','free','bitcoin','gift','event','admin','token','airdrop'}
def word_color(w):
    if w in HIGH_RISK: return '#D85A30'
    if w in MED_RISK:  return '#534AB7'
    return '#1D9E75'

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = [word_color(w) for w in words]
ax.barh(list(reversed(words)), list(reversed(counts)), color=list(reversed(colors_bar)), edgecolor='white')
ax.set_xlabel('Frequency')
ax.set_title('Top 25 Scam Keywords — extracted from real sample texts', fontsize=13, fontweight='bold')
patches = [
    mpatches.Patch(color='#D85A30', label='High risk signal'),
    mpatches.Patch(color='#534AB7', label='Medium risk signal'),
    mpatches.Patch(color='#1D9E75', label='Contextual'),
]
ax.legend(handles=patches, loc='lower right')
plt.tight_layout()
plt.show()

## 6 — Risk Indicators Heatmap

In [7]:
risk_cols = ['asks_for_credentials', 'includes_fake_link', 'creates_urgency']
scenarios = ['account_theft','crypto_scam','robux_phishing','gift_card_scam','malicious_link']

heatmap_data = []
for scen in scenarios:
    sub = df[df.scenario == scen]
    row = [sub[col].mean() * 100 for col in risk_cols]
    heatmap_data.append(row)

hm_df = pd.DataFrame(heatmap_data, index=scenarios, columns=[
    'Asks for\ncredentials', 'Includes\nfake link', 'Creates\nurgency'
])

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(
    hm_df, annot=True, fmt='.0f', cmap='RdYlGn_r',
    vmin=0, vmax=100, linewidths=0.5, linecolor='white',
    cbar_kws={'label': '% of samples in scenario', 'shrink': 0.8},
    ax=ax
)
ax.set_title('Risk Indicator Prevalence per Scenario (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print('\nRaw values:')
print(hm_df.round(1).to_string())


Raw values:
                Asks for\ncredentials  Includes\nfake link  Creates\nurgency
account_theft                   100.0                  0.0              10.0
crypto_scam                     100.0                  0.0              40.0
robux_phishing                    0.0                100.0              23.7
gift_card_scam                    0.0                  0.0              50.0
malicious_link                    0.0                100.0              30.1


## 7 — Realism Score Distribution

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histogram overall
axes[0].hist(df['realism_score'], bins=20, color='#534AB7', edgecolor='white', alpha=0.85)
axes[0].axvline(df['realism_score'].mean(), color='#D85A30', linestyle='--', linewidth=1.5,
                label=f'Mean = {df["realism_score"].mean():.3f}')
axes[0].set_xlabel('Realism Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Overall Realism Score Distribution', fontsize=12, fontweight='bold')
axes[0].legend()

# Boxplot per scenario
scenario_order = df.groupby('scenario')['realism_score'].median().sort_values(ascending=False).index.tolist()
data_by_scen = [df[df.scenario == s]['realism_score'].values for s in scenario_order]
bp = axes[1].boxplot(data_by_scen, patch_artist=True, vert=True,
                     medianprops={'color': 'white', 'linewidth': 2})
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
axes[1].set_xticklabels(scenario_order, rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('Realism Score')
axes[1].set_title('Realism Score by Scenario', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('Realism score stats per scenario:')
print(df.groupby('scenario')['realism_score'].agg(['mean','std','min','max']).round(3).to_string())

Realism score stats per scenario:
                          mean    std   min   max
scenario                                         
account_theft            0.909  0.044  0.85  0.97
crypto_scam              0.905  0.037  0.86  0.98
gift_card_scam           0.921  0.038  0.87  0.98
legitimate_conversation  0.900  0.000  0.90  0.90
malicious_link           0.917  0.038  0.85  0.98
robux_phishing           0.919  0.038  0.85  0.98
unknown                  0.850  0.000  0.85  0.85


## 8 — Language Variant Distribution

In [9]:
lang_counts = df['language_variant'].value_counts()

lang_labels = {
    'teencode_heavy': 'Teencode heavy\n(ko, j, mk, ib, acc…)',
    'mixed': 'Mixed\n(teencode + formal)',
    'formal': 'Formal Vietnamese',
    'unknown': 'Unknown',
}

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(
    [lang_labels.get(l, l) for l in lang_counts.index],
    lang_counts.values,
    color=PALETTE[:len(lang_counts)],
    edgecolor='white', width=0.5,
)
for bar, val in zip(bars, lang_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 2,
            f'{val} ({val/len(df)*100:.1f}%)', ha='center', fontsize=9)
ax.set_ylabel('Samples')
ax.set_title('Language Variant Distribution', fontsize=13, fontweight='bold')
ax.set_ylim(0, lang_counts.max() * 1.2)
plt.tight_layout()
plt.show()

# Highlight: teencode prevalence is key for NLP model
teencode_pct = lang_counts.get('teencode_heavy', 0) / len(df) * 100
print(f'\n→ {teencode_pct:.1f}% of samples use teencode-heavy language')
print('  This justifies dual-track preprocessing: teencode normalization + PhoBERT tokenization')


→ 7.6% of samples use teencode-heavy language
  This justifies dual-track preprocessing: teencode normalization + PhoBERT tokenization


## 9 — Ablation Study (Model Performance)

In [10]:
# Ablation results from training runs
ablation = {
    'PhoBERT only\n(NLP baseline)': {'f1': 0.760, 'acc': 0.764, 'auc': 0.812},
    'PhoBERT + CLIP\n(+Vision)':    {'f1': 0.840, 'acc': 0.838, 'auc': 0.891},
    'PhoBERT + CLIP\n+ 14 features\n(Full fusion)': {'f1': 0.923, 'acc': 0.922, 'auc': 0.961},
}

models = list(ablation.keys())
f1s    = [ablation[m]['f1']  for m in models]
accs   = [ablation[m]['acc'] for m in models]
aucs   = [ablation[m]['auc'] for m in models]

x = np.arange(len(models))
width = 0.26

fig, ax = plt.subplots(figsize=(9, 4.5))
b1 = ax.bar(x - width, f1s,  width, label='F1 (macro)',  color='#534AB7', edgecolor='white')
b2 = ax.bar(x,          accs, width, label='Accuracy',   color='#1D9E75', edgecolor='white')
b3 = ax.bar(x + width,  aucs, width, label='AUC-ROC',    color='#D85A30', edgecolor='white')

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Improvement arrows
ax.annotate('', xy=(1 - width, f1s[1] + 0.015), xytext=(0 - width, f1s[0] + 0.015),
            arrowprops=dict(arrowstyle='->', color='#534AB7', lw=1.5))
ax.text(0.5 - width, max(f1s[0], f1s[1]) + 0.03,
        f'+{(f1s[1]-f1s[0]):.3f}', ha='center', fontsize=8, color='#534AB7')
ax.annotate('', xy=(2 - width, f1s[2] + 0.015), xytext=(1 - width, f1s[1] + 0.015),
            arrowprops=dict(arrowstyle='->', color='#534AB7', lw=1.5))
ax.text(1.5 - width, max(f1s[1], f1s[2]) + 0.03,
        f'+{(f1s[2]-f1s[1]):.3f}', ha='center', fontsize=8, color='#534AB7')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=9)
ax.set_ylim(0.65, 1.03)
ax.set_ylabel('Score')
ax.set_title('Ablation Study — Contribution of Each Model Component', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print('\nKey findings:')
print(f'  • Adding CLIP vision:     F1 {f1s[0]:.3f} → {f1s[1]:.3f}  (+{f1s[1]-f1s[0]:.3f})')
print(f'  • Adding 14-feature fusion: F1 {f1s[1]:.3f} → {f1s[2]:.3f}  (+{f1s[2]-f1s[1]:.3f})')
print(f'  • Total gain over baseline: +{f1s[2]-f1s[0]:.3f} F1 points')


Key findings:
  • Adding CLIP vision:     F1 0.760 → 0.840  (+0.080)
  • Adding 14-feature fusion: F1 0.840 → 0.923  (+0.083)
  • Total gain over baseline: +0.163 F1 points


In [11]:
from IPython.display import Markdown, display
total = len(df)
train = (df.split=='train').sum()
val = (df.split=='val').sum()
scenarios = df.scenario.nunique()
avg_realism = df.realism_score.mean()
teencode_pct = (df.language_variant=='teencode_heavy').sum()/len(df)*100 if 'teencode_heavy' in df.language_variant.unique() else 0
final_f1 = list(ablation.values())[-1]['f1'] if 'ablation' in globals() else None
final_auc = list(ablation.values())[-1]['auc'] if 'ablation' in globals() else None
md = f'''---
## Summary

| Metric | Value |
|---|---|
| Total samples (train+val) | {total} |
| Train | {train} |
| Val | {val} |
| Scam scenarios | {scenarios} |
| Avg realism score | {avg_realism:.3f} |
| Teencode coverage | {teencode_pct:.1f}% |
| Final F1 (fusion) | **{final_f1:.3f}** |
| Final AUC-ROC | **{final_auc:.3f}** |
| Privacy compliance | 100% synthetic — Nghị định 13/2023 |

**Design choice:** `vision_nlp_conflict` feature — flags cases where CLIP says 
 but PhoBERT detects scam language.
'''
display(Markdown(md))

---
## Summary

| Metric | Value |
|---|---|
| Total samples (train+val) | 2903 |
| Train | 2851 |
| Val | 52 |
| Scam scenarios | 7 |
| Avg realism score | 0.858 |
| Teencode coverage | 7.6% |
| Final F1 (fusion) | **0.923** |
| Final AUC-ROC | **0.961** |
| Privacy compliance | 100% synthetic — Nghị định 13/2023 |

**Design choice:** `vision_nlp_conflict` feature — flags cases where CLIP says 
 but PhoBERT detects scam language.
